# FAME Database Inspection & Auditing Tools
This notebook provides a suite of tools to quickly audit, sample, and query the compiled FAME DuckDB database.

It uses `Ibis` to push computations directly to DuckDB, meaning queries execute in milliseconds and use almost zero Python memory.

In [2]:
from pathlib import Path
from dataclasses import dataclass
import os
import sys
from dotenv import load_dotenv

# Add .env's PYTHONPATH to sys.path
load_dotenv(override=True)
PYTHONPATH = os.getenv("PYTHONPATH")
if PYTHONPATH is not None:
    if PYTHONPATH not in sys.path:
        print(f"Adding {PYTHONPATH} to sys.path")
        sys.path.append(PYTHONPATH)

# Define the structure clearly
@dataclass
class Dirs:
    data_dir: Path = None       # type: ignore
    output_dir: Path = None     # type: ignore
    input_dir: Path = None      # type: ignore
dirs = Dirs()

try:
    from utils.f_0_dirs import get_data_dirs
    dirs = get_data_dirs()
    # raise ImportError("Testing ImportError for demonstration purposes") 

# For easy access purposes, if this script is run directly in a folder with the databases
# and without the dir import modules, then just set it to current_dir
except ImportError as e:
    print(f"ImportError: {e}")
    try:
        current_dir = Path(__file__).parent
    except NameError:
        current_dir = Path.cwd()
    dirs = Dirs(output_dir=current_dir, data_dir=current_dir, input_dir=current_dir)

for attr in dir(dirs):
    if attr.startswith('_'):
        continue
    if callable(getattr(dirs, attr)):
        continue
    print(f"{attr}: {getattr(dirs, attr)}")

Adding /mnt/c/Users/lazym/Documents/Code/dissertation/ to sys.path
data_dir: /mnt/h/Other computers/My computer/fame_clean/1_FAME_raw_data/2025.07.30
db_path: /mnt/c/Users/lazym/Documents/Code/dissertation/build/output/fame_data.duckdb
input_dir: /mnt/c/Users/lazym/Documents/Code/dissertation/build/input
output_dir: /mnt/c/Users/lazym/Documents/Code/dissertation/build/output
raw_data_dir: /mnt/h/Other computers/My computer/fame_clean/1_FAME_raw_data/2025.02
root_data_dir: /mnt/h/Other computers/My computer/fame_clean
root_dir: /mnt/c/Users/lazym/Documents/Code/dissertation
tmp_dir: /mnt/c/Users/lazym/Documents/Code/dissertation/build/tmp
work_dir: /mnt/c/Users/lazym/Documents/Code/dissertation/build/src


In [4]:
import pandas as pd
import ibis
import ibis.selectors as s

# 1. Setup paths
db_path = dirs.output_dir / "fame_data.duckdb"

# 2. Connect to the database
con = ibis.duckdb.connect(str(db_path))

# 3. Configure Pandas display for easier reading
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 50)

print(f"✅ Successfully connected to: {db_path}")

✅ Successfully connected to: /mnt/c/Users/lazym/Documents/Code/dissertation/build/output/fame_data.duckdb


## 1. High-Level Database Overview
Quickly check which tables exist in the database and their total row counts.

In [5]:
tables = con.list_tables()
print("📊 Database Tables Overview:\n" + "-"*30)

for table_name in tables:
    t = con.table(table_name)
    # Fast row counting pushed down to DuckDB
    row_count = t.count().execute()
    col_count = len(t.columns)
    print(f"[{table_name}]")
    print(f"   Rows: {row_count:,} | Columns: {col_count}")
print("-" * 30)

📊 Database Tables Overview:
------------------------------
[fame_derived]
   Rows: 1,942,561 | Columns: 7
[fame_fixed]
   Rows: 2,059,590 | Columns: 29
[fame_yearly]
   Rows: 24,260,361 | Columns: 25
[lars_fixed]
   Rows: 452,638 | Columns: 26
[lars_yearly]
   Rows: 2,378,089 | Columns: 35
------------------------------


In [3]:
import ibis

out_file = dirs.output_dir / "duckdb_tables.md"
con = ibis.duckdb.connect(str(dirs.output_dir / "fame_data.duckdb"))
interesting_tables = ["fame_fixed", "fame_derived", "fame_yearly", "lars_fixed", "lars_yearly"]
existing_tables = con.list_tables()
intersection = set(interesting_tables).intersection(existing_tables)

with open(out_file, "w") as f:
    f.write("# Tables in DuckDB database\n\n")
    for table in intersection:
        f.write(f"## {table}\n\n")
        f.write(f"### Number of rows: {con.table(table).count().execute():,}\n\n")
        f.write(f"### Schema:\n\n```\n{con.table(table).schema()}\n```\n\n")
        f.write(f"### Head of table:\n\n```\n{con.table(table).head().execute()}\n```\n\n")
print(f"✅ Successfully listed tables and their heads in: {out_file}")

✅ Successfully listed tables and their heads in: /mnt/c/Users/lazym/Documents/Code/dissertation/build/output/duckdb_tables.md


## 2. Fast Random Sampling (Reservoir Sampling)
Extract a random subset of rows for visual inspection. We use DuckDB's native reservoir sampling (`USING SAMPLE X ROWS`) so it returns instantly, even on tables with millions of rows, without doing a full table scan.

In [8]:
target_table = "fame_yearly"
sample_frac = 0.001  # Sample fraction for random sampling

# Native Ibis sampling using positional row count and seed parameter
table = con.table(target_table)
row_count = table.count().execute()
df_sample = table.sample(sample_frac, seed=12345).execute()

print(f"🎲 Random sample of {row_count:,} rows from '{target_table}':")
display(df_sample)

🎲 Random sample of 24,260,361 rows from 'fame_yearly':


,registered_number,year,consolidated,turnover,shareholders_funds,profit_loss_pretax,employees,tangibles,tangibles_land_and_buildings,tangibles_land_freehold,tangibles_land_leasehold,fixed_other,intangibles,fixed_total,liabilities,total_assets,liabilites_lt,cos,dividends,r_and_d,remuneration_employees,wages,social_security_costs,pensions_costs,ebitda
0,04539608,2023,False,NaN,50.046,NaN,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,04742309,2011,False,NaN,32.833,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,04980124,2022,False,NaN,890.829,NaN,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,05067241,2006,False,NaN,2.968,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,05500567,2009,False,NaN,82.033,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24242,07828876,2018,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-28.502,27.435,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
24243,11846332,2023,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-2.175
24244,03574070,2010,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-51.111,-59.000,NaN,19.296,19.296,NaN,NaN,51.464
24245,00966229,2010,None,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1171.899,-145.441,NaN,1234.933,1103.642,96.065,35.226,431.760


## 3. Specific Lookup

### Firm-specific (Search by Name or ID)
Find all fixed and derived attributes for a specific company using partial string matching (case-insensitive).

In [9]:
search_term = "TESCO"  # Can be a partial name or a Registered Number
table_fixed = con.table("fame_fixed")

# Filter where company_name contains the search term OR exactly matches registered_number
search_query = table_fixed.filter(
    table_fixed["company_name"].upper().contains(search_term.upper()) |
    (table_fixed["registered_number"] == search_term)
)

# Pull the top 5 matches
df_search_results = search_query.head(10).execute()

print(f"🔍 Top search results for '{search_term}':")
display(df_search_results[["registered_number", "company_name", "ro_address", "primary_trading_address"]])

🔍 Top search results for 'TESCO':


,registered_number,company_name,ro_address,primary_trading_address
0,10645827,BITTESCOMBE MANOR ESTATE LTD,"Bittescombe Manor Upton, Taunton, Somerset, TA...",NaN
1,10645827,BITTESCOMBE MANOR ESTATE LTD,"Bittescombe Manor Upton, Taunton, Somerset, TA...",NaN
2,11797367,TESCON CONSULTING LTD,"c/o Johnston Carmichael Office G, Ground Floor...",NaN
3,12302279,INTESCOM LTD,"34 New House, 67-68 Hatton Garden, London, EC1...",NaN
4,03527086,SITESCOPE LIMITED,"Unit 5-7, Abbey Court, Eagle Way, Sowton Indus...",NaN
5,05769298,TESCO TECH SUPPORT LIMITED,"Tesco House, Delamare Road, Cheshunt, Waltham ...","Tesco House, Delamare Road, Cheshunt, Waltham ..."
6,07374250,WHITESCOMPUTERSERVICES LIMITED,"89 The Mount, Ringwood, Hampshire, BH24 1XZ","89 The Mount, Ringwood, Hampshire, BH24 1XZ"
7,06353207,TESCO.EU.COM LIMITED,"5 Whitmore Crescent, Chelmsford, Essex, CM2 6YN","5 Whitmore Crescent, Chelmsford, Essex, CM2 6YN"
8,03091560,TESCOM (UK) SOFTWARE SYSTEMS TESTING LIMITED,"Flat 3, 80 The Lakes, Larkfield, Aylesford, Ke...",NaN
9,04460370,ELITESCORE LIMITED,"19 The Maltings, Tingewick, Buckingham, Buckin...",NaN


### Industry-specific (by 2-digit SIC or 6-digit
- 6-digit using primary_uk_sic_2007_code in fame_fixed

In [10]:
# ### Industry-specific (by 2-digit SIC or 6-digit
# - 6-digit using primary_uk_sic_2007_code in fame_fixed
import json
import ibis
import pandas as pd

# Read build/input/SIC_priorities.json
search_inds = []

get_from_json = False
if get_from_json:
    with open(dirs.input_dir / "SIC_priorities.json", "r") as f:
        sic_priorities = json.load(f)
        search_inds = [entry.get("Division") for entry in sic_priorities if entry.get("Errored") == True]
else:
    search_inds = ["70", "74"]

# Ensure strings are properly formatted (e.g. padded with zeros) just in case
search_inds_str = [str(x).zfill(2) for x in search_inds if x is not None]
print(f"Searching for following SIC divisions individually: {search_inds_str}")

# Connect to the tables in the database
t_fixed = con.table("fame_fixed")
t_derived = con.table("fame_derived")
t_yearly = con.table("fame_yearly")

# List to accumulate our row records
results = []

for ind in search_inds_str:
    # 1. Filter the fame_derived table for the specific SIC division
    regex_pattern = rf'\b{ind}\b'
    filtered_derived = t_derived.filter(
        t_derived.industry_codes.re_search(regex_pattern)
    )

    # 2. Execute the count for fame_derived
    derived_count = filtered_derived.count().execute()

    # 3. Filter fame_fixed using a semi-join
    fixed_count = t_fixed.semi_join(filtered_derived, "registered_number").count().execute()

    # 4. Filter fame_yearly using a semi-join
    yearly_count = t_yearly.semi_join(filtered_derived, "registered_number").count().execute()

    # Append the counts for this specific division to our results list
    results.append({
        "SIC_Division": ind,
        "fame_derived": derived_count,
        "fame_fixed": fixed_count,
        "fame_yearly": yearly_count
    })

# Output the results as a formatted table
output_table = pd.DataFrame(results)

print("📊 Matching Rows by Table for Each SIC Division:")
display(output_table) # use print(output_table) if you aren't in a Jupyter environment

Searching for following SIC divisions individually: ['70', '74']
📊 Matching Rows by Table for Each SIC Division:


,SIC_Division,fame_derived,fame_fixed,fame_yearly
0,70,0,0,0
1,74,235756,269162,3397496


## 4. Time-Series Construction (Joining Fixed & Yearly)
Combine the static company metadata with its longitudinal financial performance. This demonstrates the relational integrity of the `registered_number` primary key.

In [11]:
# Pick a specific company ID to track over time
selected_ids = df_sample["registered_number"].head().tolist()

t_fixed = con.table("fame_fixed")
t_yearly = con.table("fame_yearly")

# 1. Filter both tables to the target ID
firm_fixed = t_fixed.filter(t_fixed["registered_number"].isin(selected_ids))
firm_yearly = t_yearly.filter(t_yearly["registered_number"].isin(selected_ids))

# 2. Left join yearly financials onto the fixed metadata
firm_history = firm_yearly.left_join(firm_fixed, "registered_number")

# 3. Select a curated subset of columns to display
comprehensive_view = firm_history.select(
    "registered_number",
    "company_name",
    "year",
    # Safely select financial columns if they exist in the DB
    s.contains("turnover"),
    s.contains("profit_loss_pretax"),
    s.contains("employees")
).order_by("year") # Order chronologically

display(comprehensive_view.execute())

,registered_number,company_name,year,turnover,profit_loss_pretax,employees,remuneration_employees
0,05067241,SOUTH DEVON CHILLI FARM LTD,2006,NaN,NaN,NaN,NaN
1,04980124,EAST AVERCOMBE FARM LIMITED,2006,NaN,NaN,NaN,NaN
2,04980124,EAST AVERCOMBE FARM LIMITED,2006,NaN,NaN,NaN,NaN
3,05067241,SOUTH DEVON CHILLI FARM LTD,2006,NaN,NaN,NaN,NaN
4,04742309,R J BROWN LIMITED,2006,69.697,20.345,NaN,NaN
...,...,...,...,...,...,...,...
371,05500567,NEWHOUSE MILL LIMITED,2024,NaN,NaN,20.0,NaN
372,04742309,R J BROWN LIMITED,2024,NaN,NaN,2.0,NaN
373,04742309,R J BROWN LIMITED,2024,NaN,NaN,2.0,NaN
374,04980124,EAST AVERCOMBE FARM LIMITED,2024,NaN,NaN,3.0,NaN


## 5. Summary Statistics & Aggregations
Generate high-level analytical cuts (e.g., counting the number of records per year, or assessing data coverage).

In [12]:
t_yearly = con.table("fame_yearly")

# Aggregate the number of financial records available per year
# Print numbers of available records for every financial column
yearly_distribution = (
    t_yearly
    .group_by("year")
    .aggregate(
        turnover=t_yearly["turnover"].count(),
        pnl=t_yearly["profit_loss_pretax"].count(),
        employees=t_yearly["employees"].count(),
        tangibles=t_yearly["tangibles"].count(),
        t_lab=t_yearly["tangibles_land_and_buildings"].count(),
        t_land_free=t_yearly["tangibles_land_freehold"].count(),
        t_land_lease=t_yearly["tangibles_land_leasehold"].count(),
        fixed_other=t_yearly["fixed_other"].count(),
        intangibles=t_yearly["intangibles"].count(),
        fixed_total=t_yearly["fixed_total"].count(),
        liabilities=t_yearly["liabilities"].count(),
        t_assets=t_yearly["total_assets"].count(),
        liabilites_lt=t_yearly["liabilites_lt"].count(),
        cos=t_yearly["cos"].count(),
        dividends=t_yearly["dividends"].count(),
        r_and_d=t_yearly["r_and_d"].count(),
        renum=t_yearly["remuneration_employees"].count(),
        wages=t_yearly["wages"].count(),
        ss_cost=t_yearly["social_security_costs"].count(),
        pension_cost=t_yearly["pensions_costs"].count(),
        ebitda=t_yearly["ebitda"].count(),
        all=t_yearly.count()
    )
    .order_by(ibis.desc("year"))
)

print("📈 Data coverage by year:")
# Format table with commas for readability and display
count_table: pd.DataFrame = yearly_distribution.execute()
count_table_formatted = count_table.copy().reset_index(drop=True)
for col in count_table_formatted.columns:
    if col != "year":
        count_table_formatted[col] = count_table_formatted[col].apply(lambda x: f"{x:,}")
display(count_table_formatted)

📈 Data coverage by year:


,year,turnover,pnl,employees,tangibles,t_lab,t_land_free,t_land_lease,fixed_other,intangibles,fixed_total,liabilities,t_assets,liabilites_lt,cos,dividends,r_and_d,renum,wages,ss_cost,pension_cost,ebitda,all
0,2025,510,591,"5,776","3,736",252,159,106,"2,627",339,"4,909","5,651","7,664","2,856",2,0,0,0,0,0,0,2,"17,665"
1,2024,"22,940","27,791","321,668","223,352","13,453","10,137","3,967","165,623","16,260","281,307","342,014","428,595","158,810","9,179","3,852",651,"13,875","9,717","8,204","8,198","27,001","955,723"
2,2023,"60,798","76,930","566,932","427,772","32,509","22,615","11,662","311,867","42,347","554,036","666,902","836,231","317,627","27,888","10,840","2,326","36,819","30,026","25,756","25,180","74,215","1,819,052"
3,2022,"60,834","77,097","561,364","423,562","32,681","22,312","12,213","309,088","41,876","544,996","659,261","823,762","315,312","27,813","10,467","2,250","36,458","29,519","25,269","24,548","74,228","1,792,899"
4,2021,"60,069","75,457","564,720","418,512","31,775","21,597","11,857","305,375","40,738","530,023","657,533","813,890","310,177","26,427","10,024","2,128","36,163","28,727","24,261","23,427","72,763","1,769,045"
5,2020,"59,918","74,006","553,586","411,296","31,565","21,673","11,517","302,464","39,218","510,552","653,491","792,271","262,083","25,584","9,858","2,039","35,194","29,828","23,449","22,317","71,316","1,722,332"
6,2019,"58,511","71,937","419,399","399,995","32,349","23,288","10,956","302,788","38,021","490,797","628,784","758,719","219,049","24,658","10,682","1,994","33,921","33,472","22,336","20,688","69,039","1,650,901"
7,2018,"57,398","69,719","346,644","386,322","30,105","22,009","9,640","281,557","37,794","468,877","600,182","720,274","207,309","24,225","11,151","1,739","34,197","33,958","21,729","19,687","67,919","1,570,497"
8,2017,"59,963","72,622","283,465","373,862","29,260","21,740","8,960","268,151","38,282","444,530","574,719","686,140","195,181","24,953","12,542","1,680","36,913","36,649","22,257","19,290","71,784","1,504,799"
9,2016,"52,057","62,589","146,150","358,030","15,600","9,826","6,925","311,430","46,378","428,322","548,574","654,864","165,110","24,142","14,333","1,277","36,635","36,216","24,410","19,452","61,373","1,429,684"


## 6. Exporting Queries to Excel
Any queried subset of data can be instantly dumped into an Excel file for offline review.

In [13]:
import pandas as pd

approach = 'random' # first

# Dump a random sample of 500 rows from each table to a single .xlsx file in /tmp
row_count = 2000
out_file_raw = dirs.output_dir / "df_raw_sample.xlsx"
desired_tables = ["fame_derived", "fame_fixed", "fame_yearly", "lars_fixed", "lars_yearly"]

# but we want to have 5 separate tabs in the same excel file, one for each table, with the table name as the tab name
con = ibis.duckdb.connect(str(dirs.output_dir / "fame_data.duckdb"))
with pd.ExcelWriter(out_file_raw, engine='openpyxl') as writer:
    # df_raw_head.to_excel(writer, sheet_name='df_raw', index=False)
    for table in desired_tables:
        if approach == 'first':
            con.table(table).head(row_count).execute().to_excel(writer, sheet_name=table, index=False)
        elif approach == 'random':
            nb_rows = con.table(table).count().execute()
            fraction = row_count / nb_rows if nb_rows > row_count else 1.0
            df_table_rd = con.table(table).sample(fraction, seed=12345).execute()
            df_table_rd.to_excel(writer, sheet_name=table, index=False)
        
print(f"✅ Successfully dumped a random {row_count} rows of df_raw to: {out_file_raw}")

✅ Successfully dumped a random 2000 rows of df_raw to: /mnt/c/Users/lazym/Documents/Code/dissertation/build/output/df_raw_sample.xlsx
